# 🚨 GCN Crime Activity Classification — Final Pipeline (Kaggle)

**14-class classification:** Normal + 13 UCF-Crime anomaly categories.

## ✅ Prerequisites
Upload these 3 files into `/kaggle/working/` before running:
1. `tracking_and_pose.py`
2. `data_processing.py`
3. `gcn_model.py`

## 📂 Expected Dataset Structure
```
/kaggle/input/ucf-crime-dataset/
    Train/
        Normal/         *.mp4
        Abuse/          *.mp4
        Arrest/         *.mp4
        Arson/          *.mp4
        Assault/        *.mp4
        Burglary/       *.mp4
        Explosion/      *.mp4
        Fighting/       *.mp4
        RoadAccidents/  *.mp4
        Robbery/        *.mp4
        Shooting/       *.mp4
        Shoplifting/    *.mp4
        Stealing/       *.mp4
        Vandalism/      *.mp4
```

## 🗂️ Class Mapping
```
 0: Normal        1: Abuse         2: Arrest        3: Arson
 4: Assault       5: Burglary      6: Explosion     7: Fighting
 8: RoadAccidents 9: Robbery      10: Shooting     11: Shoplifting
12: Stealing     13: Vandalism
```

## 🔄 Workflow
- **Cell 1:** Install dependencies
- **Cell 2:** Paste all source code (self-contained, no separate file uploads needed)
- **Cell 3:** Feature extraction — videos → skeleton tensors → saved `.npy`
- **Cell 4:** Training setup
- **Cell 5:** Training loop with per-epoch + best-model checkpointing + resume support
- **Cell 6:** Inference helper

In [1]:
# ============================================================
# CELL 1 — FIX: Reinstall Stable PyTorch for Kaggle GPU
# ============================================================
# 1. Uninstall current versions that are causing the mismatch
!pip uninstall -y torch torchvision torchaudio ultralytics

# 2. Install compatible versions (Standard for Kaggle T4/P100)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install ultralytics deep-sort-realtime

import torch
print(f"✅ PyTorch Version: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 97.0 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.9 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 41.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 92.6 MB/s eta 0:00:0000:0100:0

In [2]:
# ============================================================
# CELL 2 — All Source Code (Updated for PNG Image Sequences)
# ============================================================

import os
import sys
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from collections import deque
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ─────────────────────────────────────────────────────────────
# Utility: IoU & Tracking Logic
# ─────────────────────────────────────────────────────────────

def compute_iou(box1, box2):
    """IoU between two [x1,y1,x2,y2] boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    if x2 < x1 or y2 < y1:
        return 0.0
    inter = (x2 - x1) * (y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return 0.0 if union == 0 else inter / union


class PoseTracker:
    def __init__(self, model_type='yolov8n-pose.pt'):
        print(f'Loading pose model: {model_type}...')
        self.model = YOLO(model_type)

    def process_frame_list(self, frame_paths):
        """Processes a list of sorted PNG image paths as a single sequence."""
        # Reset DeepSort fresh for every unique sequence to prevent ID bleed
        tracker = DeepSort(
            max_age=30,
            n_init=3,
            nms_max_overlap=1.0,
            max_cosine_distance=0.2
        )

        skeleton_sequences = {}
        
        for frame_id, img_path in enumerate(frame_paths):
            frame = cv2.imread(img_path)
            if frame is None:
                continue

            # Pose detection with YOLOv8
            results = self.model(frame, classes=0, verbose=False)
            yolo_detections = []
            yolo_keypoints  = []

            for result in results:
                boxes     = result.boxes
                keypoints = result.keypoints
                if boxes is None or keypoints is None or len(boxes) == 0:
                    continue
                for i in range(len(boxes)):
                    x1, y1, x2, y2 = map(int, boxes.xyxy[i].cpu().numpy())
                    conf = float(boxes.conf[i].cpu().numpy())
                    # DeepSort expects [x, y, w, h]
                    yolo_detections.append(([x1, y1, x2 - x1, y2 - y1], conf, 0))
                    kpts = keypoints.data[i].cpu().numpy()
                    yolo_keypoints.append({'bbox': [x1, y1, x2, y2], 'keypoints': kpts})

            # Update tracker
            tracks = tracker.update_tracks(yolo_detections, frame=frame)

            for track in tracks:
                if not track.is_confirmed():
                    continue
                tid = track.track_id
                tx1, ty1, tx2, ty2 = map(int, track.to_ltrb())

                # Match tracking box to the YOLO pose keypoints
                best_iou, best_kpts = 0, None
                for yk in yolo_keypoints:
                    iou = compute_iou([tx1, ty1, tx2, ty2], yk['bbox'])
                    if iou > best_iou:
                        best_iou  = iou
                        best_kpts = yk['keypoints']

                if best_kpts is not None and best_iou > 0.3:
                    if tid not in skeleton_sequences:
                        skeleton_sequences[tid] = []
                    skeleton_sequences[tid].append({
                        'frame_id':  frame_id,
                        'keypoints': best_kpts
                    })

        return skeleton_sequences


# ─────────────────────────────────────────────────────────────
# GCN Components (Preprocessing & Model)
# ─────────────────────────────────────────────────────────────

class SkeletonBuffer:
    def __init__(self, max_frames=30):
        self.max_frames = max_frames
        self.buffers = {}
        self.last_updated_frame = {}

    def update(self, track_id, keypoints, current_frame_idx):
        if track_id not in self.buffers:
            self.buffers[track_id] = deque(maxlen=self.max_frames)
        self.buffers[track_id].append(np.array(keypoints, dtype=np.float32))
        self.last_updated_frame[track_id] = current_frame_idx

    def get_sequence(self, track_id):
        if track_id not in self.buffers: return None
        seq = list(self.buffers[track_id])
        arr = np.array(seq)
        if len(arr) < self.max_frames:
            # Pad with first frame if sequence is too short
            pad = np.tile(arr[0], (self.max_frames - len(arr), 1, 1))
            arr = np.concatenate((pad, arr), axis=0)
        return arr[:self.max_frames]


class GCNPreprocessor:
    def __init__(self, sequence_length=30):
        self.T = sequence_length
        # Joint connections for ST-GCN
        self.bone_pairs = [(0,5),(1,0),(2,0),(3,1),(4,2),(5,0),(6,0),(7,5),(8,6),(9,7),(10,8),(11,5),(12,6),(13,11),(14,12),(15,13),(16,14)]

    def process(self, sequence):
        # Normalize and reshape to (1, 3, T, 17, 1)
        norm = sequence.copy()
        center = (norm[:, 11, :2] + norm[:, 12, :2]) / 2.0
        norm[:, :, :2] -= center[:, np.newaxis, :]
        t = np.transpose(norm, (2, 0, 1))
        return {'joint_data': np.expand_dims(np.expand_dims(t, 0), -1)}


class Graph:
    def __init__(self, strategy='spatial'):
        self.num_node = 17
        self.neighbor = [(15,13),(13,11),(16,14),(14,12),(11,12),(5,11),(6,12),(5,6),(5,7),(7,9),(6,8),(8,10),(1,2),(0,1),(0,2),(1,3),(2,4),(0,5),(0,6)]
        self.A = self.get_adjacency()

    def get_adjacency(self):
        A = np.zeros((3, self.num_node, self.num_node))
        # Simple spatial partitioning logic
        for i, j in self.neighbor:
            A[0, i, i] = 1
            A[1, i, j] = A[1, j, i] = 1
        return A


class STGCN_Block(nn.Module):
    def __init__(self, in_channels, out_channels, K=3, stride=1):
        super().__init__()
        self.sgcn = nn.Conv2d(in_channels, out_channels * K, kernel_size=1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, (9,1), (stride,1), (4,0))
        )
        self.residual = nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, (stride,1)), nn.BatchNorm2d(out_channels)) if in_channels != out_channels or stride != 1 else lambda x: x

    def forward(self, x, A):
        res = self.residual(x)
        n, c, t, v = x.size()
        x = self.sgcn(x).view(n, A.size(0), -1, t, v)
        x = torch.einsum('nkctv,kvw->nctw', x, A)
        return self.tcn(x) + res


class ActionRecognitionGCN(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        self.graph = Graph()
        self.A = nn.Parameter(torch.tensor(self.graph.A, dtype=torch.float32), requires_grad=False)
        self.data_bn = nn.BatchNorm1d(3 * 17)
        self.layers = nn.ModuleList([
            STGCN_Block(3, 64),
            STGCN_Block(64, 128, stride=2),
            STGCN_Block(128, 256, stride=2)
        ])
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        n, c, t, v, m = x.size()
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(n * m, v * c, t)
        x = self.data_bn(x).view(n * m, v, c, t).permute(0, 2, 3, 1)
        for layer in self.layers: x = layer(x, self.A)
        x = F.avg_pool2d(x, x.size()[2:]).view(n, m, -1).mean(dim=1)
        return self.fc(x)


print('✅ All classes loaded successfully.')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'   Device: {device}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ All classes loaded successfully.
   Device: cuda


In [3]:
# ============================================================
# CELL 3 — Balanced Feature Extraction (FIXED: Frame Cap + Verbose Logging)
# ============================================================
from collections import defaultdict
import re
import time

CLASSES = ['NormalVideos', 'Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion',
           'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
CLASS_TO_IDX    = {cls: idx for idx, cls in enumerate(CLASSES)}
DATASET_ROOT    = '/kaggle/input/datasets/odins0n/ucf-crime-dataset/Train'
SEQUENCE_LEN    = 30
MAX_SEQ_PER_CLASS   = 10     # sequences per class
MAX_FRAMES_PER_VIDEO = 60   # ← FIX: cap frames (only need SEQUENCE_LEN * 2)

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def extract_features_from_list(frame_paths, tracker, vid_label=""):
    # ── FIX: Subsample frames evenly instead of processing all ──
    if len(frame_paths) > MAX_FRAMES_PER_VIDEO:
        indices = np.linspace(0, len(frame_paths) - 1, MAX_FRAMES_PER_VIDEO, dtype=int)
        frame_paths = [frame_paths[i] for i in indices]
        print(f'         ↳ Subsampled to {len(frame_paths)} frames')

    buffer = SkeletonBuffer(max_frames=SEQUENCE_LEN)
    preprocessor = GCNPreprocessor(sequence_length=SEQUENCE_LEN)
    seqs = tracker.process_frame_list(frame_paths)
    if not seqs:
        print(f'         ⚠️  No pose detected in: {vid_label}')
        return None
    longest_id = max(seqs, key=lambda k: len(seqs[k]))
    for data in seqs[longest_id]:
        buffer.update(longest_id, data['keypoints'], data['frame_id'])
    raw_tensor = buffer.get_sequence(longest_id)
    return preprocessor.process(raw_tensor)['joint_data'] if raw_tensor is not None else None


def construct_dataset(data_dir, tracker):
    data_tensors, labels = [], []
    skipped = 0
    total_start = time.time()

    for class_name in CLASSES:
        class_idx  = CLASS_TO_IDX[class_name]
        class_path = os.path.join(data_dir, class_name)
        if not os.path.exists(class_path):
            print(f'⚠️  Skipping {class_name}: Path not found'); continue

        all_images   = glob.glob(os.path.join(class_path, "*.png"))
        video_groups = defaultdict(list)
        for img in all_images:
            video_id = "_".join(os.path.basename(img).split("_")[:-1])
            video_groups[video_id].append(img)

        balanced_keys = list(video_groups.keys())[:MAX_SEQ_PER_CLASS]
        print(f'\n[{class_idx:2d}] {class_name:<15}: {len(balanced_keys)} sequences | '
              f'{sum(len(video_groups[k]) for k in balanced_keys)} total frames')

        for seq_num, vid_id in enumerate(balanced_keys, 1):
            frames = video_groups[vid_id]
            frames.sort(key=natural_sort_key)
            raw_count = len(frames)

            print(f'      [{seq_num}/{len(balanced_keys)}] {vid_id} — {raw_count} frames raw', end='', flush=True)
            t0 = time.time()

            result = extract_features_from_list(frames, tracker, vid_label=vid_id)

            elapsed = time.time() - t0
            if result is not None:
                data_tensors.append(result[0])
                labels.append(class_idx)
                print(f'  ✅  ({elapsed:.1f}s)')
            else:
                skipped += 1
                print(f'  ❌ skipped ({elapsed:.1f}s)')

    total_elapsed = time.time() - total_start
    print(f'\n{"─"*60}')
    print(f'⏱️  Total extraction time : {total_elapsed/60:.1f} min')
    print(f'   Sequences extracted   : {len(data_tensors)}')
    print(f'   Sequences skipped     : {skipped}')

    if not data_tensors: return None, None
    return np.stack(data_tensors), np.array(labels)


# ── RUN ──
_TRACKER = PoseTracker('yolov8n-pose.pt')
X_numpy, Y_numpy = construct_dataset(DATASET_ROOT, _TRACKER)

if X_numpy is not None:
    np.save('/kaggle/working/X_data.npy', X_numpy)
    np.save('/kaggle/working/Y_labels.npy', Y_numpy)
    print(f'💾 Dataset Saved: {len(X_numpy)} sequences.')


Loading pose model: yolov8n-pose.pt...

[ 0] NormalVideos   : 10 sequences | 205264 total frames
      [1/10] Normal_Videos642_x264 — 5826 frames raw         ↳ Subsampled to 60 frames


/usr/local/lib/python3.12/dist-packages/deep_sort_realtime/embedder/embedder_pytorch.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.lo

         ⚠️  No pose detected in: Normal_Videos642_x264
  ❌ skipped (2.3s)
      [2/10] Normal_Videos288_x264 — 353 frames raw         ↳ Subsampled to 60 frames
  ✅  (2.1s)
      [3/10] Normal_Videos307_x264 — 62802 frames raw         ↳ Subsampled to 60 frames
  ✅  (1.7s)
      [4/10] Normal_Videos557_x264 — 1841 frames raw         ↳ Subsampled to 60 frames
         ⚠️  No pose detected in: Normal_Videos557_x264
  ❌ skipped (1.4s)
      [5/10] Normal_Videos630_x264 — 6339 frames raw         ↳ Subsampled to 60 frames
  ✅  (2.0s)
      [6/10] Normal_Videos308_x264 — 97651 frames raw         ↳ Subsampled to 60 frames
         ⚠️  No pose detected in: Normal_Videos308_x264
  ❌ skipped (1.8s)
      [7/10] Normal_Videos533_x264 — 7933 frames raw         ↳ Subsampled to 60 frames
  ✅  (1.8s)
      [8/10] Normal_Videos195_x264 — 476 frames raw         ↳ Subsampled to 60 frames
  ✅  (2.2s)
      [9/10] Normal_Videos947_x264 — 18082 frames raw         ↳ Subsampled to 60 frames
  ✅  (2.4s)
      

In [4]:
# ============================================================
# CELL 4 — Training Setup (FIXED)
# ============================================================

NUM_CLASSES  = 14          # ← FIX 1: was missing
SEQUENCE_LEN = 30

X_NPY_PATH = '/kaggle/working/X_data.npy'
Y_NPY_PATH = '/kaggle/working/Y_labels.npy'

if os.path.exists(X_NPY_PATH) and os.path.exists(Y_NPY_PATH):
    print('📂 Loading extracted features from cache...')
    X_numpy = np.load(X_NPY_PATH)
    Y_numpy = np.load(Y_NPY_PATH)
    print(f'   X shape: {X_numpy.shape}  |  Y shape: {Y_numpy.shape}')
else:
    print('⚠️  No .npy cache found. Using DUMMY DATA for dry run.')
    print('   Run Cell 3 extraction first for real training.')
    X_numpy = np.random.rand(80, 3, SEQUENCE_LEN, 17, 1).astype(np.float32)
    Y_numpy = np.random.randint(0, NUM_CLASSES, 80).astype(np.int64)

# ── DataLoaders ──
X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
Y_tensor = torch.tensor(Y_numpy, dtype=torch.long)
dataset  = TensorDataset(X_tensor, Y_tensor)

train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'\n   Train samples : {train_size}')
print(f'   Val   samples : {val_size}')

# ── Model ──                          ← FIX 2: removed in_channels=3
model     = ActionRecognitionGCN(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5, factor=0.5
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Model ready — trainable params: {total_params:,}')
print(f'   Adjacency A shape : {model.A.shape}  (K=3 spatial partitions)')


📂 Loading extracted features from cache...
   X shape: (84, 3, 30, 17, 1)  |  Y shape: (84,)

   Train samples : 67
   Val   samples : 17

✅ Model ready — trainable params: 946,484
   Adjacency A shape : torch.Size([3, 17, 17])  (K=3 spatial partitions)


In [5]:
# ============================================================
# CELL 5 — Training Loop
#
# Checkpoints saved:
#   checkpoints/epoch_XXX.pth     — every epoch (full resumable state)
#   checkpoints/latest_checkpoint.pth — always the most recent epoch
#   checkpoints/best_checkpoint.pth   — best val_acc (full resumable state)
#   best_gcn_weights.pth               — best weights only  (for inference)
#   final_gcn_weights.pth              — last epoch weights  (for inference)
#   class_mapping.npy                  — class index array
#
# To RESUME after a crash: just re-run this cell.
# It will auto-detect latest_checkpoint.pth and pick up from there.
# ============================================================

EPOCHS              = 60
EARLY_STOP_PATIENCE = 10
CHECKPOINT_DIR      = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_acc     = 0.0
patience_counter = 0
start_epoch      = 0

# ── Auto-resume from latest checkpoint if it exists ──
resume_path = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
if os.path.exists(resume_path):
    print(f'🔄 Resuming from: {resume_path}')
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch      = ckpt['epoch'] + 1
    best_val_acc     = ckpt['best_val_acc']
    patience_counter = ckpt['patience_counter']
    print(f'   ↳ Resumed at epoch {start_epoch} | Best val acc so far: {best_val_acc:.2f}%')
else:
    print('🚀 Starting fresh training...')

print(f'   Checkpoint dir : {CHECKPOINT_DIR}')
print(f'   Epochs         : {start_epoch} → {EPOCHS}')
print(f'   Early stop     : {EARLY_STOP_PATIENCE} epochs patience')
print('─' * 75)


for epoch in range(start_epoch, EPOCHS):

    # ────────────────── Train ──────────────────
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted  = torch.max(outputs.data, 1)
        total        += labels.size(0)
        correct      += (predicted == labels).sum().item()

    train_acc  = 100.0 * correct / total
    avg_loss   = running_loss / len(train_loader)

    # ────────────────── Validate ──────────────────
    model.eval()
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total   += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc    = 100.0 * val_correct / val_total if val_total > 0 else 0.0
    current_lr = optimizer.param_groups[0]['lr']

    print(f'Epoch [{epoch+1:3d}/{EPOCHS}]  '
          f'Loss: {avg_loss:.4f}  '
          f'Train: {train_acc:.2f}%  '
          f'Val: {val_acc:.2f}%  '
          f'LR: {current_lr:.6f}')

    # Step LR scheduler
    scheduler.step(val_acc)

    # ────────────────── Build checkpoint dict ──────────────────
    checkpoint = {
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_acc':         best_val_acc,
        'patience_counter':     patience_counter,
        'train_acc':            train_acc,
        'val_acc':              val_acc,
        'loss':                 avg_loss,
    }

    # ── Save: numbered epoch snapshot ──
    epoch_path = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch+1:03d}.pth')
    torch.save(checkpoint, epoch_path)

    # ── Save: overwrite latest (used for resume) ──
    torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth'))

    # ── Save: best model ──
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        checkpoint['best_val_acc'] = best_val_acc  # update inside dict too
        torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, 'best_checkpoint.pth'))
        torch.save(model.state_dict(), '/kaggle/working/best_gcn_weights.pth')
        print(f'   ✅ Best model saved  → val_acc: {best_val_acc:.2f}%')
    else:
        patience_counter += 1
        print(f'   ⏳ No improvement ({patience_counter}/{EARLY_STOP_PATIENCE})')
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'\n⏹  Early stopping triggered at epoch {epoch+1}.')
            break

print(f'\n{'─'*75}')
print(f'✅ Training complete!  Best val acc: {best_val_acc:.2f}%')

# ── Final outputs ──
torch.save(model.state_dict(), '/kaggle/working/final_gcn_weights.pth')
np.save('/kaggle/working/class_mapping.npy', np.array(CLASSES))

print('\n📦 Output files:')
print(f'   /kaggle/working/best_gcn_weights.pth     ← best val acc weights (inference)')
print(f'   /kaggle/working/final_gcn_weights.pth    ← last epoch weights   (inference)')
print(f'   /kaggle/working/class_mapping.npy        ← class index array')
print(f'   {CHECKPOINT_DIR}/best_checkpoint.pth    ← best full state      (resumable)')
print(f'   {CHECKPOINT_DIR}/epoch_XXX.pth          ← per-epoch snapshots  (resumable)')
print('\nDownload from the Kaggle Output pane ↗')

🚀 Starting fresh training...
   Checkpoint dir : /kaggle/working/checkpoints
   Epochs         : 0 → 60
   Early stop     : 10 epochs patience
───────────────────────────────────────────────────────────────────────────
Epoch [  1/60]  Loss: 7.9163  Train: 5.97%  Val: 5.88%  LR: 0.005000
   ✅ Best model saved  → val_acc: 5.88%
Epoch [  2/60]  Loss: 6.6219  Train: 10.45%  Val: 11.76%  LR: 0.005000
   ✅ Best model saved  → val_acc: 11.76%
Epoch [  3/60]  Loss: 5.3935  Train: 5.97%  Val: 5.88%  LR: 0.005000
   ⏳ No improvement (1/10)
Epoch [  4/60]  Loss: 4.5794  Train: 11.94%  Val: 11.76%  LR: 0.005000
   ⏳ No improvement (2/10)
Epoch [  5/60]  Loss: 3.8675  Train: 14.93%  Val: 11.76%  LR: 0.005000
   ⏳ No improvement (3/10)
Epoch [  6/60]  Loss: 3.4224  Train: 11.94%  Val: 5.88%  LR: 0.005000
   ⏳ No improvement (4/10)
Epoch [  7/60]  Loss: 2.9103  Train: 8.96%  Val: 5.88%  LR: 0.005000
   ⏳ No improvement (5/10)
Epoch [  8/60]  Loss: 3.2326  Train: 10.45%  Val: 0.00%  LR: 0.005000
   ⏳ 

In [9]:
# ============================================================
# CELL 6 — Inference Helper
# Run after training to test predictions on individual videos.
#
# ── [FIX 5] tensor_obj is already (1,3,T,17,1) — no unsqueeze ──
# ============================================================

def load_model_for_inference(weights_path, num_classes=NUM_CLASSES, device=device):
    """Load a saved model for inference."""
    m = ActionRecognitionGCN(num_classes=num_classes, in_channels=3).to(device)
    m.load_state_dict(torch.load(weights_path, map_location=device))
    m.eval()
    print(f'✅ Model loaded from: {weights_path}')
    return m


def predict_video(video_path, model, device=device, classes=CLASSES):
    """
    Predict crime class for a single video.
    Returns: (predicted_class_name, probabilities_array)
    """
    if not os.path.exists(video_path):
        print(f'❌ Video not found: {video_path}')
        return None

    # Use a fresh tracker instance for inference too
    infer_tracker = PoseTracker('yolov8n-pose.pt')
    result = extract_features_from_video(video_path, infer_tracker)

    if result is None:
        print('⚠️  No person/pose detected in video.')
        return None

    # result shape: (1, 3, T, 17, 1) — already has batch dim
    # No unsqueeze needed  ← [FIX 5]
    x = torch.tensor(result, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_idx   = probs.argmax()
    pred_class = classes[pred_idx]
    confidence = probs[pred_idx] * 100

    print(f'\n🎯 Prediction : {pred_class}  ({confidence:.1f}%)')
    print('   Top-3 predictions:')
    for i in probs.argsort()[::-1][:3]:
        bar = '█' * int(probs[i] * 20)
        print(f'   {classes[i]:<15} {probs[i]*100:5.1f}%  {bar}')

    return pred_class, probs


# ── Example usage (uncomment to run) ──
best_model = load_model_for_inference('/kaggle/working/best_gcn_weights.pth')
predict_video('/kaggle/input/datasets/odins0n/ucf-crime-dataset/Train/Fighting/Fighting002_x264_1200.png', best_model)

print('✅ Inference helper ready.')
print('   Uncomment the two lines above to run a test prediction.')

TypeError: ActionRecognitionGCN.__init__() got an unexpected keyword argument 'in_channels'